<a href="https://colab.research.google.com/github/ibnaziarafi/vs-code-coding/blob/main/Small_GPT_PyTorch_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Small GPT with built-in PyTorch transformer blocks

Run cells from top to bottom in Google Colab. Select a GPU runtime if available; CPU execution is supported but slower. Colab normally already includes PyTorch, so no installation cell is needed.

This is a small GPT-style, character-level causal language model trained from random parameters. It follows the learning objective used in Karpathy's video: predict the next character at every input position. It is not a GPT-2 checkpoint or an instruction-following chatbot.

**Architecture:** token embeddings + learned positions → 2 masked transformer blocks → final LayerNorm → vocabulary scores. Each block includes 4 attention heads, the Q/K/V projections, attention output projection, a 64→256→64 ReLU MLP, residual connections, LayerNorm, and dropout.

PyTorch calls the reused component `TransformerEncoderLayer`. It has self-attention and an MLP without a cross-attention module. Supplying a causal mask makes it suitable as the block in our GPT-style causal stack. We create each block independently so they do not start as identical cloned layers.

The default run downloads Tiny Shakespeare (about 1 MB) unless `input.txt` already exists, uses a 64-character context, and trains for 1,000 updates. Text quality depends on training progress; sensible sentences are not guaranteed. Increase STEPS after confirming the basic run works. The final 10% of the text is reserved for validation, not used for parameter updates. Repeated tuning on that validation split is not an independent final test.

Generation samples probabilistically. Temperature below 1 sharpens the distribution; above 1 flattens it. Once the sequence exceeds 64 characters, generation retains the newest 64 as context and reuses positions 0 through 63. There is no KV cache in this beginner implementation.

All nonblank code lines have explanatory comments. During preparation, syntax and notebook structure were checked. PyTorch is unavailable in the preparation environment, so training was not executed there. Cell 4 executes shape and future-token-leakage checks in your Colab session before training.


## 1. Setup and model size


In [ ]:
import torch  # Import tensors, GPU support, and automatic gradients.
from torch import nn  # Import neural-network components.
from torch.nn import functional as F  # Import cross-entropy loss and softmax.
from pathlib import Path  # Work with the downloaded text file.
from urllib.request import urlopen  # Download the training text without an API key.

torch.manual_seed(42)  # Choose repeatable random starting values; GPU results can still vary.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use a Colab GPU when available.
BLOCK_SIZE = 128  # Allow each prediction to use at most 64 characters of context.
BATCH_SIZE = 16  # Train on 16 text chunks per update.
WIDTH = 64  # Represent each character position with 64 numbers.
HEADS = 4  # Use four attention heads, each with 16 query/key/value features.
LAYERS = 2  # Stack two transformer blocks.
STEPS = 1000  # Start with 1000 updates; increase later for a longer learning experiment.
LEARNING_RATE = 0.001  # Choose AdamW's update learning rate.
print("Device:", device, "| PyTorch:", torch.__version__)  # Display the active device and library version.


Device: cuda | PyTorch: 2.11.0+cu130


## 2. Download text, tokenize, and create shifted targets


In [ ]:
path = Path("input.txt")  # Reuse a local text file if one is already present in this Colab session.
if not path.exists():  # Download Tiny Shakespeare only when the file is missing.
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"  # Use Karpathy's public text dataset.
    with urlopen(url, timeout=60) as response:  # Open the dataset download with a timeout.
        path.write_bytes(response.read())  # Save the downloaded text locally in Colab.
text = path.read_text(encoding="utf-8")  # Read the training corpus as one string.
chars = sorted(set(text))  # Build a vocabulary containing each distinct character.
stoi = {char: index for index, char in enumerate(chars)}  # Map characters to token IDs.
encode = lambda string: [stoi[char] for char in string]  # Convert a string into character IDs.
decode = lambda ids: "".join(chars[index] for index in ids)  # Convert IDs back into readable text.
vocab_size = len(chars)  # Count the possible character tokens.
data = torch.tensor(encode(text), dtype=torch.long)  # Encode the complete corpus as integers on CPU.
split = int(0.9 * len(data))  # Reserve the final 10 percent for validation.
train_data, val_data = data[:split], data[split:]  # Keep training and validation text separate.
assert min(len(train_data), len(val_data)) > BLOCK_SIZE  # Require enough text in both splits for shifted targets.
print("Characters:", len(data), "| Vocabulary:", vocab_size)  # Display corpus and vocabulary sizes.

def get_batch(source):  # Sample several fixed-length contexts from one data split.
    starts = torch.randint(len(source) - BLOCK_SIZE, (BATCH_SIZE,))  # Choose valid random chunk starts.
    x = torch.stack([source[i:i + BLOCK_SIZE] for i in starts])  # Read the input characters.
    y = torch.stack([source[i + 1:i + BLOCK_SIZE + 1] for i in starts])  # Shift forward by one character to get every next-token target.
    return x.to(device), y.to(device)  # Move this batch to the active device.


Characters: 1115394 | Vocabulary: 65


## 3. Build a small GPT-style causal transformer


In [ ]:
class SmallGPT(nn.Module):  # Define a causal next-character model using built-in transformer blocks.
    def __init__(self):  # Create the trainable components.
        super().__init__()  # Initialise PyTorch's model machinery.
        self.token = nn.Embedding(vocab_size, WIDTH)  # Learn one WIDTH-number vector per character.
        self.position = nn.Embedding(BLOCK_SIZE, WIDTH)  # Learn position vectors within the context window.
        self.blocks = nn.ModuleList([  # Create independently initialised blocks; their weights are not shared.
            nn.TransformerEncoderLayer(  # Reuse a self-attention block; an explicit causal mask gives it GPT-style information flow.
                d_model=WIDTH, nhead=HEADS, dim_feedforward=4 * WIDTH,  # Set representation width, attention-head count, and MLP hidden width.
                dropout=0.1, activation="relu", batch_first=True, norm_first=True,  # Include dropout, ReLU, batch-first tensors, and pre-normalisation.
            ) for _ in range(LAYERS)  # Build the requested number of separate blocks.
        ])  # Register every block so its parameters are trained and saved.
        self.norm = nn.LayerNorm(WIDTH)  # Normalise the final token representations.
        self.output = nn.Linear(WIDTH, vocab_size)  # Produce a vocabulary score at every token position.
        mask = torch.triu(torch.full((BLOCK_SIZE, BLOCK_SIZE), float("-inf")), diagonal=1)  # Put negative infinity above the diagonal to block future tokens.
        self.register_buffer("mask", mask)  # Store the non-trainable mask and move it with the model.

    def forward(self, ids):  # Turn a batch of token IDs into next-token scores at every position.
        length = ids.shape[1]  # Read the current number of input token positions.
        assert 0 < length <= BLOCK_SIZE  # Keep the sequence within the learned position table.
        positions = torch.arange(length, device=ids.device)  # Number positions from zero.
        x = self.token(ids) + self.position(positions)  # Combine token identity and order; shape [batch, time, WIDTH].
        for block in self.blocks:  # Process the sequence through each transformer block.
            x = block(x, src_mask=self.mask[:length, :length])  # Apply masked attention, residual paths, normalisation, and the MLP.
        return self.output(self.norm(x))  # Return raw scores of shape [batch, time, vocabulary].

model = SmallGPT().to(device)  # Initialise the model and place it on the selected device.
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))  # Count its adjustable numbers.


Trainable parameters: 116673


## 4. Check shapes and confirm future tokens are blocked


In [ ]:
model.eval()  # Disable dropout so the causality check compares deterministic forward passes.
with torch.no_grad():  # Run the checks without storing gradients.
    x, y = get_batch(train_data)  # Obtain a small batch for inspection.
    original = x[:2, :8].clone()  # Keep two examples and their first eight tokens.
    changed = original.clone()  # Copy the examples so their final token can be changed.
    changed[:, -1] = (changed[:, -1] + 1) % vocab_size  # Alter only the final input token in each example.
    before = model(original)  # Calculate scores from the original examples.
    after = model(changed)  # Calculate scores after changing the future token.
    assert before.shape == (2, 8, vocab_size)  # Require a vocabulary prediction at every input position.
    assert torch.allclose(before[:, :-1], after[:, :-1], atol=1e-5, rtol=1e-5)  # Earlier predictions must not depend on the changed future token.
print("Shape check passed; causal-mask check passed.")  # Confirm that Colab executed both checks successfully.
print("Input sample:", repr(decode(x[0, :12].tolist())))  # Show a short input fragment, including special characters visibly.
print("Shifted targets:", repr(decode(y[0, :12].tolist())))  # Show its one-character-ahead targets.


Shape check passed; causal-mask check passed.
Input sample: 'sabel, a sis'
Shifted targets: 'abel, a sist'


## 5. Train with separate validation measurements


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)  # Update all trainable tables and layers.

@torch.no_grad()  # Evaluate without recording gradient graphs.
def estimate_loss(source):  # Average validation or training loss over several fresh batches.
    previous_mode = model.training  # Remember whether the model was training before this measurement.
    model.eval()  # Disable dropout during evaluation.
    losses = []  # Collect the loss from each evaluation batch.
    for _ in range(10):  # Average ten batches for a less noisy measurement.
        x, y = get_batch(source)  # Sample from the requested split.
        logits = model(x)  # Predict at every position without updating weights.
        losses.append(F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1)).item())  # Treat each position as one classification example.
    model.train(previous_mode)  # Restore the model's previous training or evaluation mode.
    return sum(losses) / len(losses)  # Return average loss across the evaluation batches.

model.train()  # Enable dropout for training.
for step in range(1, STEPS + 1):  # Repeat the training update.
    x, y = get_batch(train_data)  # Use only the training split for parameter updates.
    optimizer.zero_grad()  # Clear gradients from the previous update.
    logits = model(x)  # Run a forward pass for all positions in the batch.
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))  # Compare raw vocabulary scores with shifted target IDs.
    loss.backward()  # Calculate gradients through the complete model.
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Limit the overall gradient size to reduce excessively large updates.
    optimizer.step()  # Apply AdamW's parameter updates.
    if step == 1 or step % 200 == 0 or step == STEPS:  # Report progress at the start, periodically, and at the end.
        print(f"Step {step}: train={estimate_loss(train_data):.3f}, validation={estimate_loss(val_data):.3f}")  # Measure both splits without training on validation text.


Step 1: train=4.245, validation=4.260
Step 200: train=2.561, validation=2.558
Step 400: train=2.487, validation=2.499
Step 600: train=2.434, validation=2.461
Step 800: train=2.404, validation=2.429
Step 1000: train=2.330, validation=2.377


## 6. Generate text and save the trained model


In [ ]:
@torch.no_grad()  # Generate without calculating gradients.
def generate(prompt, new_characters=300, temperature=0.8):  # Continue a supplied prompt one character at a time.
    if not prompt or any(char not in stoi for char in prompt):  # Require a nonempty prompt made of known characters.
        raise ValueError("Use a nonempty prompt containing only characters in the training vocabulary.")  # Explain unsupported prompts.
    if temperature <= 0:  # Prevent division by zero or a reversed probability preference.
        raise ValueError("Temperature must be greater than zero.")  # Explain the valid sampling setting.
    model.eval()  # Disable dropout while generating.
    ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)  # Encode the prompt as one sequence.
    for _ in range(new_characters):  # Append one sampled character during each iteration.
        context = ids[:, -BLOCK_SIZE:]  # Keep only the most recent context that fits the model.
        logits = model(context)[:, -1, :] / temperature  # Use the final position's next-character scores and adjust sampling sharpness.
        probabilities = F.softmax(logits, dim=-1)  # Convert raw scores into a next-character probability distribution.
        next_id = torch.multinomial(probabilities, num_samples=1)  # Sample one character rather than always choosing the largest score.
        ids = torch.cat([ids, next_id], dim=1)  # Append the sampled character to the sequence.
    return decode(ids[0].tolist())  # Decode the prompt and generated continuation together.

print(generate("ROMEO:\n", new_characters=300))  # Generate a short continuation; an early small model may produce rough text.
config = dict(block_size=BLOCK_SIZE, width=WIDTH, heads=HEADS, layers=LAYERS, dropout=0.1, activation="relu")  # Record the architecture needed to rebuild the model.
torch.save({"model_state": model.state_dict(), "chars": chars, "config": config}, "small_gpt.pth")  # Save weights, tokenizer vocabulary, and architecture in the Colab session.
print("Saved small_gpt.pth in the Colab Files panel; download it if you want to keep the trained weights.")  # Explain where the generated checkpoint is stored.


ROMEO:
Odshe imeam pore n hous o bead heal thice thes am'dsu th.


HAnd s le myond, wing se, thaverain fon herens pus where, le prorere my am thither so my
RI the the Burellld su sreat are bllsas.

LI Thalllcchr upof conel lise hose isthe ve by speceas hishoy sas thor douthe heeey med
Th thalour acough cob
Saved small_gpt.pth in the Colab Files panel; download it if you want to keep the trained weights.


In [ ]:
model.train()  # Enable training behaviour, including dropout.

for step in range(1001, 10000):  # Continue from update 1001 through update 3000.
    x, y = get_batch(train_data)  # Sample input text and shifted next-character targets.
    optimizer.zero_grad()  # Clear previous gradients.
    logits = model(x)  # Predict the next character at every input position.
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))  # Compare predictions with correct character IDs.
    loss.backward()  # Calculate gradients throughout the model.
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Limit excessively large gradients.
    optimizer.step()  # Update the existing model parameters.

    if step % 200 == 0:  # Check progress every 200 updates.
        train_loss = estimate_loss(train_data)  # Measure training performance without updating weights.
        val_loss = estimate_loss(val_data)  # Measure performance on separate validation text.
        print(f"Step {step}: train={train_loss:.3f}, validation={val_loss:.3f}")  # Display both losses.

print(generate("ROMEO:\n", new_characters=300, temperature=0.8))  # Generate another sample with the same sampling setting.

Step 1200: train=2.267, validation=2.277
Step 1400: train=2.184, validation=2.227
Step 1600: train=2.127, validation=2.151
Step 1800: train=2.076, validation=2.107
Step 2000: train=2.033, validation=2.083
Step 2200: train=2.003, validation=2.059
Step 2400: train=1.990, validation=2.037
Step 2600: train=1.943, validation=1.997
Step 2800: train=1.908, validation=1.988
Step 3000: train=1.886, validation=1.988
Step 3200: train=1.874, validation=1.988
Step 3400: train=1.852, validation=1.972
Step 3600: train=1.838, validation=1.958
Step 3800: train=1.831, validation=1.913
Step 4000: train=1.781, validation=1.911
Step 4200: train=1.791, validation=1.918
Step 4400: train=1.769, validation=1.912
Step 4600: train=1.780, validation=1.887
Step 4800: train=1.726, validation=1.869
Step 5000: train=1.776, validation=1.890
Step 5200: train=1.755, validation=1.874
Step 5400: train=1.726, validation=1.851
Step 5600: train=1.717, validation=1.838
Step 5800: train=1.708, validation=1.847
Step 6000: train

In [ ]:
for group in optimizer.param_groups:  # Keep the existing optimizer and its accumulated state.
    group["lr"] = 0.0002  # Reduce the learning rate from 0.001 to 0.0002.

print("Before:", "train", estimate_loss(train_data), "validation", estimate_loss(val_data))  # Record a baseline.

model.train()  # Enable training behaviour.

for extra_step in range(1, 2000):  # Perform 1000 additional updates.
    x, y = get_batch(train_data)  # Sample training contexts and next-character targets.
    optimizer.zero_grad()  # Clear previous gradients.
    logits = model(x)  # Predict at every token position.
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))  # Measure prediction mistakes.
    loss.backward()  # Calculate gradients.
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Limit excessive gradient sizes.
    optimizer.step()  # Apply the smaller updates.

    if extra_step % 200 == 0:  # Check progress every 200 updates.
        print(f"Extra update {extra_step}: train={estimate_loss(train_data):.3f}, validation={estimate_loss(val_data):.3f}")  # Report both losses.

print(generate("ROMEO:\n", new_characters=300, temperature=0.8))  # Keep sampling temperature unchanged for comparison.

Before: train 1.590164065361023 validation 1.729042887687683
Extra update 200: train=1.611, validation=1.753
Extra update 400: train=1.554, validation=1.757
Extra update 600: train=1.598, validation=1.765
Extra update 800: train=1.564, validation=1.770
Extra update 1000: train=1.591, validation=1.753
Extra update 1200: train=1.576, validation=1.749
Extra update 1400: train=1.564, validation=1.733
Extra update 1600: train=1.572, validation=1.751
Extra update 1800: train=1.581, validation=1.776
ROMEO:
And I am king, pretch my rueds of lord,
And this a prepeak to thou dese;
Of thy a stry heart some, shall so, and you battle thy
is the pronce not my you water and again.

KING EDWARD IV:
Rome house I can him with prince leter,
Marryiar I make it steed couse countremy.

RUTIO:
My have will the we car
